# Baseline strategies

No-action and fixed-schedule baselines evaluated under the same battery and accounting assumptions.


## Fair comparison rules

Both strategies use the same battery and accounting assumptions:

| Assumption | Value |
|---|---:|
| Battery capacity | 2.0 MWh |
| Operating SoC range | 10% to 90% of capacity |
| Charge/discharge power limit | 1.0 MW |
| Time step | 1 hour |
| Charge efficiency | 92% |
| Discharge efficiency | 92% |
| Wear allowance | 5 EUR/MWh discharged |
| Initial and terminal SoC | 0.8 MWh |

The fixed rule requests charging at 02:00–03:00 and discharging at 17:00–18:00. Its charging request is reduced automatically if the maximum SoC is reached. The sellable energy is calculated from the energy actually accepted during those charging hours and then distributed across the predefined discharge hours. This enforces the terminal SoC condition without using realised prices to choose the schedule.

In [1]:
from pathlib import Path

import math
import pandas as pd

prices = pd.read_csv(
    Path("data/belgium_day_ahead_sample.csv"),
    parse_dates=["timestamp"],
).sort_values("timestamp").reset_index(drop=True)

CAPACITY_MWH = 2.0
POWER_LIMIT_MW = 1.0
MIN_SOC_MWH = 0.10 * CAPACITY_MWH
MAX_SOC_MWH = 0.90 * CAPACITY_MWH
CHARGE_EFFICIENCY = 0.92
DISCHARGE_EFFICIENCY = 0.92
WEAR_EUR_PER_MWH_DISCHARGED = 5.0
STEP_HOURS = 1.0
INITIAL_SOC_MWH = 0.8

prices.head()


,timestamp,price_eur_per_mwh
0,2025-01-06 00:00:00+01:00,61.2
1,2025-01-06 01:00:00+01:00,57.8
2,2025-01-06 02:00:00+01:00,52.1
3,2025-01-06 03:00:00+01:00,48.7
4,2025-01-06 04:00:00+01:00,45.9


## Dispatch and accounting function


In [2]:
def apply_action(soc_mwh, requested_charge_mwh, requested_discharge_mwh):
    """Apply one hourly action while enforcing battery energy and power limits."""
    max_grid_energy_mwh = POWER_LIMIT_MW * STEP_HOURS

    # Clamp headroom to zero to prevent floating-point boundary noise from creating negative actions.
    available_charge_headroom_mwh = max(MAX_SOC_MWH - soc_mwh, 0.0)
    charge_mwh = min(
        max(requested_charge_mwh, 0.0),
        max_grid_energy_mwh,
        available_charge_headroom_mwh / CHARGE_EFFICIENCY,
    )

    soc_after_charge_mwh = soc_mwh + CHARGE_EFFICIENCY * charge_mwh

    # Limit discharge by power and energy available above the minimum SoC.
    available_discharge_energy_mwh = max(soc_after_charge_mwh - MIN_SOC_MWH, 0.0)
    discharge_mwh = min(
        max(requested_discharge_mwh, 0.0),
        max_grid_energy_mwh,
        available_discharge_energy_mwh * DISCHARGE_EFFICIENCY,
    )

    next_soc_mwh = soc_after_charge_mwh - discharge_mwh / DISCHARGE_EFFICIENCY
    return charge_mwh, discharge_mwh, next_soc_mwh


def evaluate_fixed_rule(price_frame, charge_hours, discharge_hours):
    """Evaluate a predefined schedule with a terminal-SoC-consistent discharge request."""
    soc_mwh = INITIAL_SOC_MWH
    records = []

    # First determine how much energy the requested charging hours can physically accept.
    planned_charge_mwh = []
    planning_soc_mwh = INITIAL_SOC_MWH
    for row in price_frame.itertuples(index=False):
        requested_charge_mwh = POWER_LIMIT_MW * STEP_HOURS if row.timestamp.hour in charge_hours else 0.0
        charge_mwh, _, planning_soc_mwh = apply_action(planning_soc_mwh, requested_charge_mwh, 0.0)
        planned_charge_mwh.append(charge_mwh)

    # Distribute the sellable energy evenly across the predefined discharge hours.
    total_sellable_energy_mwh = sum(planned_charge_mwh) * CHARGE_EFFICIENCY * DISCHARGE_EFFICIENCY
    discharge_energy_per_hour_mwh = (
        total_sellable_energy_mwh / len(discharge_hours) if discharge_hours else 0.0
    )

    for row in price_frame.itertuples(index=False):
        hour = row.timestamp.hour
        requested_charge_mwh = POWER_LIMIT_MW * STEP_HOURS if hour in charge_hours else 0.0
        requested_discharge_mwh = discharge_energy_per_hour_mwh if hour in discharge_hours else 0.0

        charge_mwh, discharge_mwh, next_soc_mwh = apply_action(
            soc_mwh, requested_charge_mwh, requested_discharge_mwh
        )

        purchase_cost_eur = row.price_eur_per_mwh * charge_mwh
        sales_revenue_eur = row.price_eur_per_mwh * discharge_mwh
        wear_cost_eur = WEAR_EUR_PER_MWH_DISCHARGED * discharge_mwh

        records.append({
            "timestamp": row.timestamp,
            "price_eur_per_mwh": row.price_eur_per_mwh,
            "charge_mwh": charge_mwh,
            "discharge_mwh": discharge_mwh,
            "soc_mwh": next_soc_mwh,
            "purchase_cost_eur": purchase_cost_eur,
            "sales_revenue_eur": sales_revenue_eur,
            "wear_cost_eur": wear_cost_eur,
            "contribution_eur": sales_revenue_eur - purchase_cost_eur - wear_cost_eur,
        })
        soc_mwh = next_soc_mwh

    return pd.DataFrame(records)


## Run the baselines


In [3]:
no_action_log = evaluate_fixed_rule(prices, charge_hours=set(), discharge_hours=set())
fixed_rule_log = evaluate_fixed_rule(prices, charge_hours={2, 3}, discharge_hours={17, 18})

def summarise_strategy(name, log):
    return {
        "strategy": name,
        "purchase_cost_eur": log["purchase_cost_eur"].sum(),
        "sales_revenue_eur": log["sales_revenue_eur"].sum(),
        "wear_cost_eur": log["wear_cost_eur"].sum(),
        "contribution_eur": log["contribution_eur"].sum(),
        "starting_soc_mwh": INITIAL_SOC_MWH,
        "ending_soc_mwh": log["soc_mwh"].iloc[-1],
    }

summary = pd.DataFrame([
    summarise_strategy("No action", no_action_log),
    summarise_strategy("Fixed-time rule", fixed_rule_log),
]).set_index("strategy")

summary.round(2)


,purchase_cost_eur,sales_revenue_eur,wear_cost_eur,contribution_eur,starting_soc_mwh,ending_soc_mwh
strategy,,,,,,
No action,0.00,0.00,0.0,0.00,0.8,0.8
Fixed-time rule,56.33,110.91,4.6,49.97,0.8,0.8


## Validate both strategies


In [4]:
for name, log in {"No action": no_action_log, "Fixed-time rule": fixed_rule_log}.items():
    assert (log["charge_mwh"] >= 0).all()
    assert (log["discharge_mwh"] >= 0).all()
    assert (log["charge_mwh"] <= POWER_LIMIT_MW * STEP_HOURS + 1e-9).all()
    assert (log["discharge_mwh"] <= POWER_LIMIT_MW * STEP_HOURS + 1e-9).all()
    assert (log["soc_mwh"] >= MIN_SOC_MWH - 1e-9).all()
    assert (log["soc_mwh"] <= MAX_SOC_MWH + 1e-9).all()
    assert math.isclose(log["soc_mwh"].iloc[-1], INITIAL_SOC_MWH, abs_tol=1e-9), (
        f"{name} does not end at the required SoC"
    )

print("All baseline validation checks passed.")


All baseline validation checks passed.


## Inspect the fixed-rule actions


In [5]:
fixed_rule_log.loc[
    (fixed_rule_log["charge_mwh"] > 0) | (fixed_rule_log["discharge_mwh"] > 0),
    ["timestamp", "price_eur_per_mwh", "charge_mwh", "discharge_mwh", "soc_mwh", "contribution_eur"],
]


,timestamp,price_eur_per_mwh,charge_mwh,discharge_mwh,soc_mwh,contribution_eur
2,2025-01-06 02:00:00+01:00,52.1,1.000000,0.00,1.72,-52.100000
3,2025-01-06 03:00:00+01:00,48.7,0.086957,0.00,1.80,-4.234783
17,2025-01-06 17:00:00+01:00,112.4,0.000000,0.46,1.30,49.404000
18,2025-01-06 18:00:00+01:00,128.7,0.000000,0.46,0.80,56.902000
